# M0.1 — Induction proofs of loop invariants, as property-based tests

Derive item M0.1.d3, build item M0.1.b1. Each algorithm carries a loop invariant
proved by induction on the iteration count. The same invariant is then asserted
inside the loop and hammered with Hypothesis: proof and test are the same object.

Structure of every proof:

* **Base:** the invariant holds before the first iteration.
* **Step:** if it holds before iteration *k*, it holds after it.
* **Exit:** invariant + loop guard false ⇒ the postcondition.

In [ ]:
from collections import Counter

from hypothesis import given, settings, strategies as st

## 1. Running sum

```
s = 0
for k in 1..n: s += k
```

**Invariant** $I(k)$: after iteration $k$, $s = k(k+1)/2$.

* Base, $k = 0$: $s = 0 = 0\cdot 1/2$.
* Step: assume $s = k(k+1)/2$. Iteration $k+1$ adds $k+1$, so
  $s = k(k+1)/2 + (k+1) = (k+1)(k+2)/2$, which is $I(k+1)$.
* Exit at $k = n$: $s = n(n+1)/2$.

In [ ]:
def running_sum(n: int) -> int:
    s = 0
    assert s == 0  # I(0)
    for k in range(1, n + 1):
        s += k
        assert 2 * s == k * (k + 1), (k, s)  # I(k)
    return s


@settings(max_examples=200)
@given(st.integers(min_value=0, max_value=5_000))
def test_running_sum(n):
    assert 2 * running_sum(n) == n * (n + 1)


test_running_sum()
print("running sum: invariant held on every example")

## 2. Insertion sort

**Invariant** $J(i)$: before the outer iteration with index $i$, the prefix `a[:i]`
is sorted and is a permutation of the original `xs[:i]`; the suffix `a[i:]` is untouched.

* Base, $i = 1$: a one-element prefix is sorted.
* Step: the inner loop shifts every element of the sorted prefix greater than
  `key` one place right and drops `key` into the hole, so `a[:i+1]` is sorted and
  holds exactly the old `a[:i]` plus `xs[i]`.
* Exit at $i = n$: the whole list is a sorted permutation of `xs`.

In [ ]:
def insertion_sort(xs: list[int]) -> list[int]:
    a = list(xs)
    for i in range(1, len(a)):
        # J(i)
        assert all(a[j] <= a[j + 1] for j in range(i - 1))
        assert Counter(a[:i]) == Counter(xs[:i]) and a[i:] == xs[i:]
        key, j = a[i], i - 1
        while j >= 0 and a[j] > key:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key
    return a


@settings(max_examples=300)
@given(st.lists(st.integers(min_value=-1000, max_value=1000), max_size=60))
def test_insertion_sort(xs):
    out = insertion_sort(xs)
    assert out == sorted(xs)


test_insertion_sort()
print("insertion sort: invariant held on every example")

## 3. Euclid's gcd

**Invariant** $G$: $\gcd(a, b) = \gcd(a_0, b_0)$ at the top of every iteration.

* Base: $a = a_0,\ b = b_0$.
* Step: any common divisor of $a$ and $b$ divides $a \bmod b = a - qb$, and any
  common divisor of $b$ and $a \bmod b$ divides $a = qb + (a \bmod b)$. Same divisors, same gcd.
* Exit at $b = 0$: $\gcd(a, 0) = a$.
* Termination: $b$ strictly decreases and stays $\ge 0$ (well-ordering).

In [ ]:
import math


def gcd(a0: int, b0: int) -> int:
    a, b = a0, b0
    target = math.gcd(a0, b0)  # oracle, only used to check G
    prev_b = None
    while b != 0:
        assert math.gcd(a, b) == target  # G
        assert prev_b is None or b < prev_b  # variant decreases
        prev_b = b
        a, b = b, a % b
    return a


@settings(max_examples=300)
@given(st.integers(min_value=0, max_value=10**12), st.integers(min_value=0, max_value=10**12))
def test_gcd(a, b):
    g = gcd(a, b)
    assert g == math.gcd(a, b)
    if g:
        assert a % g == 0 and b % g == 0


test_gcd()
print("gcd: invariant and variant held on every example")

## What a failing invariant looks like

Break the step (off-by-one in the inner loop guard) and Hypothesis finds a
minimal counterexample. The cell catches the failure so the notebook still runs top to bottom.

In [ ]:
def broken_sort(xs):
    a = list(xs)
    for i in range(1, len(a)):
        key, j = a[i], i - 1
        while j > 0 and a[j] > key:  # bug: should be j >= 0
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key
    return a


@settings(max_examples=300, database=None)
@given(st.lists(st.integers(), max_size=20))
def test_broken(xs):
    assert broken_sort(xs) == sorted(xs)


try:
    test_broken()
except AssertionError:
    print("Hypothesis found a counterexample, as expected")
else:
    raise RuntimeError("expected the broken sort to fail")